# Day 7 build-along: HOW MUCH DO COSTS EAT?
Uses your calculator (nse_costs.py) on REAL NIFTYBEES data.
A toy 50/200-day moving-average long/flat rule is used ONLY as a "something that trades" example.
It is NOT an idea we're testing. Still, log it in your trial log (honesty is a habit).
Parts: A. trade count and gross result. B. Net of statutory costs. C. Plus spread (low/high). D. Plus tax.
E. The same costs if the rule traded 10x more often. F. Capital size: why Rs 10,000 accounts suffer.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from nse_costs import round_trip, after_tax_gain
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path(".")
b = pd.read_csv(HERE.parent / "03_data" / "NIFTYBEES_daily.csv", index_col=0, parse_dates=True)
px = b["Close"].where(~b.index.isin(pd.to_datetime(["2019-12-19", "2019-12-20"]))).dropna().loc["2010":]
# Bad-tick filter: a one-day spike of >8% that fully reverses the next day is a data error, not a market move.
lr = np.log(px).diff()
spike = (lr.abs() > 0.08) & (lr.shift(-1).abs() > 0.08) & (np.sign(lr) != np.sign(lr.shift(-1)))
print("Bad ticks removed:", [d.date().isoformat() for d in px.index[spike]], "(Day 14's audit will hunt these properly)")
px = px[~spike]
r = px.pct_change().fillna(0)

def run(px, fast, slow):
    sig = (px.rolling(fast).mean() > px.rolling(slow).mean()).astype(int).shift(1).fillna(0)  # trade next day: no look-ahead
    return sig

def backtest(sig, capital=1e5, spread_bps=0.0, tax=False):
    """Long/flat, full capital when long (1.0x, no leverage). sig[d] = position held DURING day d
    (already shifted, so it was decided at the previous close). Costs charged on each round trip at exit."""
    eq = capital; long = False; pos_val = 0.0; entry_val = entry_day = None
    trades = 0; cost_total = tax_total = 0.0; curve = []
    for d, s_, ret in zip(sig.index, sig.values, r.values):
        if s_ == 1 and not long:                       # enter at the previous close
            long = True; pos_val = eq; entry_val = eq; entry_day = d
        elif s_ == 0 and long:                         # exit at the previous close
            c = round_trip(entry_val, pos_val, "delivery", "etf_equity")["total"] + (entry_val + pos_val) * spread_bps / 2 / 1e4
            gain = pos_val - entry_val - c
            t = (gain - after_tax_gain(gain, (d - entry_day).days)) if (tax and gain > 0) else 0.0
            eq = pos_val - c - t; long = False; trades += 1; cost_total += c; tax_total += t
        if long:
            pos_val *= (1 + ret)
        curve.append(pos_val if long else eq)
    curve = pd.Series(curve, index=sig.index)
    yrs = (curve.index[-1] - curve.index[0]).days / 365.25
    return dict(cagr=(curve.iloc[-1] / capital) ** (1 / yrs) - 1, trades=trades, cost=cost_total, tax=tax_total, curve=curve)

sig = run(px, 50, 200)
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

## A-D. One rule, four levels of honesty

In [ ]:
print("=== A-D. 50/200 MA long/flat on NIFTYBEES, 2010-2026, Rs 1 lakh, 1.0x ===")
cases = {"A. gross (no costs)": dict(), "B. + statutory costs": dict(), "C1. + spread 1 bps": dict(spread_bps=1),
         "C2. + spread 10 bps": dict(spread_bps=10), "D. + spread 10 bps + tax": dict(spread_bps=10, tax=True)}
import nse_costs
res = {}
for name, kw in cases.items():
    if name.startswith("A"):
        orig = nse_costs.round_trip
        backtest.__globals__["round_trip"] = lambda *a, **k: {"total": 0.0}
        res[name] = backtest(sig, **kw)
        backtest.__globals__["round_trip"] = orig
    else:
        res[name] = backtest(sig, **kw)
    x = res[name]; print(f"  {name:28s} CAGR {x['cagr']*100:6.2f}%  round trips {x['trades']:3d}  costs Rs {x['cost']:8.0f}  tax Rs {x['tax']:8.0f}")
bh = (px.iloc[-1] / px.iloc[0]) ** (365.25 / (px.index[-1] - px.index[0]).days) - 1
print(f"  Buy and hold NIFTYBEES, same cleaned data (no costs or tax shown): CAGR {bh*100:.2f}%")
print("  Note: an open position at the end is marked to market, untaxed and unsold (flatters every row equally).")
for name, x in res.items(): ax[0].plot(x["curve"], label=name)
ax[0].set_yscale("log"); ax[0].legend(fontsize=7); ax[0].set_title("A-D. Same rule, four levels of honesty")

## E. Make it trade more: 5/20 MA (a "fast" version)

In [ ]:
print("\n=== E. Faster rule (5/20 MA) ===")
fast = run(px, 5, 20)
for name, kw in {"gross": None, "statutory": dict(), "+10 bps spread": dict(spread_bps=10), "+10 bps + tax": dict(spread_bps=10, tax=True)}.items():
    if kw is None:
        backtest.__globals__["round_trip"] = lambda *a, **k: {"total": 0.0}; x = backtest(fast); backtest.__globals__["round_trip"] = nse_costs.round_trip
    else:
        x = backtest(fast, **kw)
    print(f"  {name:16s} CAGR {x['cagr']*100:6.2f}%  round trips {x['trades']}")
    ax[1].plot(x["curve"], label=name)
ax[1].set_yscale("log"); ax[1].legend(fontsize=8); ax[1].set_title("E. A faster rule pays the toll more often")

## F. Account size: the fixed DP charge

In [ ]:
print("\n=== F. Round-trip cost in bps by trade size (NIFTYBEES delivery, statutory + DP, no spread) ===")
sizes = [2_000, 5_000, 10_000, 25_000, 50_000, 1_00_000, 5_00_000]
bps = [round_trip(v, v, "delivery", "etf_equity")["bps_of_buy_value"] for v in sizes]
for v, x in zip(sizes, bps): print(f"  Rs {v:>8,}: {x:6.1f} bps")
ax[2].plot(sizes, bps, "o-"); ax[2].set_xscale("log"); ax[2].set_xlabel("trade size (Rs)"); ax[2].set_ylabel("bps per round trip")
ax[2].set_title("F. Small accounts pay more per rupee")
plt.tight_layout(); plt.savefig(HERE / "cost_drag_lab.png", dpi=105); print("\nSaved cost_drag_lab.png")

## Questions for your log
1. How many CAGR points did costs + tax take from the 50/200 rule? From the 5/20 rule? Why so different?
2. Which single cost line matters most for NIFTYBEES delivery at Rs 1 lakh? At Rs 5,000?
3. Why is tax a "cost" that depends on holding period, and what does that do to fast strategies?
4. Why did we shift the signal by one day? What would happen to the gross CAGR without it, and why is that a lie?
5. Both rules trail buy-and-hold by ~6 CAGR points BEFORE costs. What does that tell you about "it has a positive CAGR"
   as evidence of a strategy? (Benchmark first, always.)
6. Log both rules in trial-log.md (N += 2). Neither is a strategy we're pursuing.